**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/huszar-2022"], check=True)
    os.chdir("/content/encore/entries/huszar-2022")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Huszar et al. (2022): the earlier attempt's findings, checked on DANDI:000552

Attempt `huszar-2022-02`, made blind, is compared with `huszar-2022-01` in the reconciliation block of
`attempts/huszar-2022-02.yaml` (docs/protocol.md, "Repeat attempts"). `huszar-2022-01` was made on version
0.230630.2304. Findings 12 and 14 of `huszar-2022-01` are reported again by `huszar-2022-02` and are not
checked here; findings 7 and 13 are also reported again, and are looked at here only for the part that may
have changed. This notebook checks the others directly on
[DANDI:000552](https://dandiarchive.org/dandiset/000552/0.261113.1035), version 0.261113.1035, streaming
the 104 processed files and the raw files' metadata with remfile. It uses the entry's helper module
`huszar.py` for the asset list and for the metadata of each processed file.

In [ ]:
import collections

import numpy as np
import pandas as pd
import requests

import huszar as h

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
print("dandiset", h.DANDISET, "version", h.VERSION)
S = h.survey(workers=12)
print(len(S), "processed files")

## Data check

In [ ]:
D = pd.DataFrame([dict(subject=m["subject_id"], session=m["session_id"], units=m["n_units"],
                       sex=m["sex"], age=m["age"]) for m in S])
print(D.groupby("subject").agg(files=("session", "size"), units=("units", "sum")).to_string())
print("subjects", D.subject.nunique(), "files", len(D), "units", D.units.sum())

## Earlier finding 1: light-responsive labels and pulse times

> No unit is labelled light-responsive (birthdated), and no optogenetic pulse times are stored in the processed or the raw files, so the paper's comparisons of same- and different-birthdate neurons and its birthdate profiles (22 of the 35 claims listed here) cannot be tested. The labels exist in the lab's CellExplorer files (optoTag, putativeCellType), which are not public at a pinned identifier.

In [ ]:
lr = sum(int(np.sum(m.get("u_light_responsive", []))) for m in S)
print("files with a light_responsive column:", sum("light_responsive" in m["u_columns"] for m in S), "of", len(S))
print("files with opto_p_salt and opto_h_reliable:",
      sum({"opto_p_salt", "opto_h_reliable"} <= set(m["u_columns"]) for m in S))
print("units labelled light-responsive, all files:", lr)
lr96 = sum(int(np.sum(m.get("u_light_responsive", []))) for m in S
           if m["subject_id"] not in h.HOMECAGE_EXCLUDED)
print("units labelled light-responsive, the 15 homecage mice:", lr96)
print("files with OptogeneticPulses:", sum(m["n_pulses"] > 0 for m in S), "; pulses in all:", sum(m["n_pulses"] for m in S))

Every processed file now has a `light_responsive` column, with the lab's statistics (`opto_p_salt`,
`opto_h_reliable`) beside it, and an `OptogeneticPulses` table. The problem is gone.

## Earlier finding 2: cell types and waveforms

> Units carry no putative cell type and no waveforms, though the paper classifies 7,411 pyramidal cells and 2,060 interneurons.

In [ ]:
ct = collections.Counter(x for m in S for x in m.get("u_putative_cell_type", []))
print("putative_cell_type over all units:", dict(ct))
ct96 = collections.Counter(x for m in S if m["subject_id"] not in h.HOMECAGE_EXCLUDED
                           for x in m.get("u_putative_cell_type", []))
print("in the 15 homecage mice:", dict(ct96))
print("files with waveform_mean:", sum(m["wf_shape"] is not None for m in S), "of", len(S))

Units now carry a putative cell type and a mean waveform in every file. The problem is gone.

## Earlier finding 3: mice without units or position

> Six of the 17 mice (e13_1f1, e13_1m1, e14_1m1, e15_4m1, e15_4m2, e15_5m1; 39 files) have sleep states and ripples but no units and no position. The dandiset holds 8,201 units where the paper reports 9,471.

In [ ]:
six = ["e13_1f1", "e13_1m1", "e14_1m1", "e15_4m1", "e15_4m2", "e15_5m1"]
print(D[D.subject.isin(six)].groupby("subject").agg(files=("session", "size"), units=("units", "sum")).to_string())
print("files with units:", int((D.units > 0).sum()), "of", len(D), "; units in all:", D.units.sum())
print("units in the 15 homecage mice:", D[~D.subject.isin(h.HOMECAGE_EXCLUDED)].units.sum())
pos = {m["subject_id"] for m in S if m["series"]}
print("mice of the six with position:", sorted(pos & set(six)))

The six mice now have units in all of their files. They have no position, which the paper does not
lead one to expect (they are not among the maze mice of Supplementary Table 1), so that half of the
statement was not a problem of the files. The units of the problem are back, so it is gone.

## Earlier finding 4: novel explorations without position

> Of the 16 novel environment explorations, only 9 have position and trials, and no open field has position. The novel-cohort sessions hold 7 further epochs of 10 to 50 min without position or video that may be the missing explorations.

In [ ]:
rows = []
for m in S:
    if m["subject_id"] not in h.NOVEL_MICE:
        continue
    for k, v in m["series"].items():
        if k.startswith("SubjectPosition/") and "tmaze_familiar" not in k:
            rows.append(dict(session=m["session_id"], series=k.split("/")[1], unit=v["unit"], n=v["n"],
                             t0=round(v["t0"]), t1=round(v["t1"])))
N = pd.DataFrame(rows)
N["environment"] = N.series.str.split("_", n=1).str[1]
print(N.to_string(index=False))
print("novel-environment position series:", len(N), "; open fields:", int(N.environment.str.startswith("openField").sum()))
print("environments:", sorted(N.environment.unique()))

The novel-cohort sessions hold 16 position series outside the familiar maze, 7 of them in the three
open fields, so every one of the 16 explorations now has position. The problem is gone. (One of them,
the linear_manu series of e16_3m2_211211, has timestamps inside the open-field epoch; `huszar-2022-02`
reports that as its finding 2.)

## Earlier finding 5: subject sex differs between files

> The subject sex differs between files of the same mouse. e13_1f1 is Female in 3 files and Male in 2; e13_1m1 is Male in 9 and Female in 3. The value in most of each mouse's files gives the paper's 12 males and 5 females.

In [ ]:
sx = D.groupby("subject").sex.agg(lambda s: dict(collections.Counter(s)))
print(sx.to_string())
print("mice whose files disagree on sex:", int((D.groupby("subject").sex.nunique() > 1).sum()))
by_mouse = D.groupby("subject").sex.first()
print("males", int((by_mouse == "M").sum()), "females", int((by_mouse == "F").sum()))

Every mouse now has one sex in all of its files, and they give 12 males and 5 females. The problem is
gone.

## Earlier finding 6: is_familiar_maze

> The trials column is_familiar_maze is True only for the first two trials of sessions with a single maze recording, where every trial is on the familiar maze (112 True, 1,182 False).

In [ ]:
T = []
for m in S:
    if m["trials"]:
        t = pd.DataFrame(m["trials"])
        t["session"] = m["session_id"]
        T.append(t)
T = pd.concat(T, ignore_index=True)
T["single_recording_session"] = T.session.map(T.groupby("session").recordings.nunique() == 1)
print(pd.crosstab(T.single_recording_session, T.is_familiar_maze).to_string())
first = T.groupby("session").recordings.transform("min")
print(pd.crosstab(T.recordings == first, T.is_familiar_maze, rownames=["first recording"]).to_string())

In sessions with a single maze recording every trial is now `is_familiar_maze` True, and the column
is True on the first recording of every session. The problem is gone.

## Earlier finding 7: novel positions in 0 to 1 labelled cm

> In several novel recordings (e13_26m1 211019, 211116, 211119; e15_13f1 220120; e16_3m2 211213) x and y lie between 0 and 1 though stated in cm.

`huszar-2022-02` reports the missing scale of these positions as its finding 3; this section looks
only at the unit they are labelled with.

In [ ]:
print(N[N.session.isin(["e13_26m1_211019", "e13_26m1_211116", "e13_26m1_211119", "e15_13f1_220120",
                        "e16_3m2_211213"])][["session", "series", "unit"]].to_string(index=False))
print("units of the novel position series:", dict(collections.Counter(N.unit)))
m = next(m for m in S if m["session_id"] == "e13_26m1_211019")
print(m["series"]["SubjectPosition/SpatialSeries2_tmaze_ipshita"]["desc"])

The series are no longer labelled cm: their unit is "n.a." and their descriptions say they are in the
lab's units with no scale given. The mislabelling is fixed; the scale itself is still missing, as
`huszar-2022-02` reports.

## Earlier finding 8: session start of the raw files

> Twelve of the 13 raw files give a session start of 2011-08-18 for sessions recorded in 2021 and 2022.

In [ ]:
out, url = [], f"{h.API}/dandisets/{h.DANDISET}/versions/{h.VERSION}/assets/?page_size=200"
while url:
    r = requests.get(url, timeout=60).json()
    out += r["results"]
    url = r["next"]
raw = sorted([a for a in out if not a["path"].endswith("behavior+ecephys.nwb")], key=lambda a: a["path"])
rows = []
for a in raw:
    with h.open_nwb(a) as f:
        rows.append(dict(file=a["path"].split("/")[-1], session_start=h._s(f["session_start_time"][()]),
                         acquisition=",".join(f["acquisition"].keys())))
R = pd.DataFrame(rows)
print(R.to_string(index=False))
R["date_in_name"] = R.file.str.extract(r"-(\d{6})-raw")[0]
R["date_in_start"] = R.session_start.str[2:10].str.replace("-", "")
print("raw files:", len(R), "; start date matching the file name:", int((R.date_in_name == R.date_in_start).sum()))

The version holds 11 raw files, and each gives the date of its file name as its session start. The
problem is gone.

## Earlier finding 9: the LFP of e16_3m1_210201

> The processed file of e16_3m1_210201 holds a copy of the 1250 Hz LFP of the raw file, labelled with a rate of 30,000 Hz and a conversion factor of 3.05e-13 V.

In [ ]:
m = next(m for m in S if m["session_id"] == "e16_3m1_210201")
a = next(a for a in h.assets() if "e16-3m1-210201" in a["path"])
with h.open_nwb(a) as f:
    lfp = f["processing/ecephys/LFP/LFP"]
    print("rate", float(lfp["starting_time"].attrs["rate"]), "Hz; conversion",
          float(lfp["data"].attrs["conversion"]), lfp["data"].attrs["unit"], "; shape", lfp["data"].shape)
rates = collections.Counter(m.get("lfp_rate") for m in S)
print("LFP rates over the processed files:", dict(rates))

The LFP of this file is labelled 1250 Hz with a conversion factor of 1.95e-07 V, as in the other
files. The problem is gone.

## Earlier finding 10: units without a link to the electrodes

> Units have no link to the electrodes table, and their group_id values ("Group3") do not match the electrode group names ("Group 3") in 64 of 65 files, so a unit's shank and position on the probe cannot be recovered.

In [ ]:
rows = []
for a in h.assets():
    with h.open_nwb(a) as f:
        u = f["units"]
        el = f["general/extracellular_ephys/electrodes"]
        n_el = el["id"].shape[0]
        has_link = "electrodes" in u and "electrodes_index" in u
        ok_idx = has_link and int(u["electrodes"][:].max()) < n_el
        groups = set(k for k in f["general/extracellular_ephys"].keys() if k != "electrodes")
        refs = {f[r].name.split("/")[-1] for r in u["electrode_group"][:]} if "electrode_group" in u else set()
        rows.append(dict(file=a["path"].split("/")[-1], electrodes_link=has_link, index_in_range=ok_idx,
                         group_refs_resolve=bool(refs) and refs <= groups))
L = pd.DataFrame(rows)
print(L[["electrodes_link", "index_in_range", "group_refs_resolve"]].sum().to_string(), "of", len(L))

Every units table now has an `electrodes` column into the electrodes table and an `electrode_group`
that refers to the file's electrode groups, so a unit's shank and place on the probe can be recovered.
The problem is gone.

## Earlier finding 11: the embryonic day

> The embryonic day of electroporation, the variable the paper is about, is stored only as the prefix of each subject id (e13 for E13.5); the age is given as P3M/P6M for every mouse.

In [ ]:
a = h.assets()[0]
with h.open_nwb(a) as f:
    sub = f["general/subject"]
    for k in sub:
        print(k, ":", h._s(sub[k][()]))
print("ages over the files:", dict(collections.Counter(m["age"] for m in S)))

The embryonic day is still held only in the subject id, and the age is still P3M/P6M in every file,
but the subject description now gives the mapping (the e-number of the id is the electroporation day)
and says that the source has no other record of the day or of each mouse's age. What the finding
asked for, a stated meaning for the prefix, is now in the files, so the problem is gone.

## Earlier finding 13: trial counts and missing trial times

> Only 38 of 65 familiar-maze sessions have ten or more trials on each visited arm, and 6 have fewer than 20 trials, where the Methods require ten per arm in every session; 19 of 1,942 trials have no start or stop time.

`huszar-2022-02` reports the trial counts as its finding 6; this section looks at the missing times.

In [ ]:
print("trials:", len(T), "in", T.session.nunique(), "sessions; with no start or stop time:",
      int((T.start_time.isna() | T.stop_time.isna()).sum()))

The 19 trials with no start or stop time are still there.

## Earlier finding 15: the novel-environment cohort

> The paper calls the novel-environment cohort of three mice separate, but one of them (e13_26m1) is also among the nine figure-eight mice, in the dandiset and in the paper's code.

In [ ]:
rows = []
for m in S:
    if m["subject_id"] not in h.NOVEL_MICE:
        continue
    keys = list(m["series"])
    rows.append(dict(session=m["session_id"], familiar_maze=any("tmaze_familiar" in k for k in keys),
                     novel=any(k.startswith("SubjectPosition/") and "tmaze_familiar" not in k for k in keys)))
C = pd.DataFrame(rows)
C["mouse"] = C.session.str.rsplit("_", n=1).str[0]
print(C.groupby("mouse")[["familiar_maze", "novel"]].sum().to_string())
fam = {m["subject_id"] for m in S
       if any("tmaze_familiar" in k or "figure-8" in v["desc"] for k, v in m["series"].items())}
print("mice with a familiar figure-eight subsession:", sorted(fam))
print("novel-cohort mice among the nine of the familiar-maze analyses (huszar.FAMILIAR_MICE):",
      sorted(h.NOVEL_MICE & h.FAMILIAR_MICE))

All three novel-cohort mice run the familiar figure-eight first in each session. e13_26m1 is also one
of the nine mice of the familiar-maze analyses (the Figure 5 column of Supplementary Table 1), so the
cohort the paper calls separate shares a mouse with the figure-eight cohort; the inconsistency is
still there.